# Scout 04 — Visual evidence
Pass networks, shot maps, radar comparison. Reads MARTS (not raw).

In [ ]:
PROJECT_ID="statsbomb-football-iq"
!pip install -q mplsoccer
from google.colab import auth; auth.authenticate_user()
from google.cloud import bigquery; import pandas as pd
bq=bigquery.Client(project=PROJECT_ID)
def q(s): return bq.query(s).to_dataframe()

In [ ]:
# ---- PASS NETWORK for any team in any match ----
TEAM="Spain"; MATCH=None   # set MATCH to a match_id, or None to aggregate
from mplsoccer import Pitch
import matplotlib.pyplot as plt
where=f"team_name='{TEAM}'"+(f" AND match_id={MATCH}" if MATCH else "")
net=q(f'''SELECT passer_name,receiver_name,SUM(passes) passes,
          AVG(avg_start_x) sx,AVG(avg_start_y) sy
          FROM `{PROJECT_ID}.statsbomb_marts.mart_team_pass_network`
          WHERE {where} GROUP BY 1,2''')
pos=net.groupby('passer_name').agg(x=('sx','mean'),y=('sy','mean'),
        vol=('passes','sum')).reset_index()
pitch=Pitch(pitch_type='statsbomb',pitch_color='#1a1a2e',line_color='#888')
fig,ax=pitch.draw(figsize=(11,7)); fig.set_facecolor('#1a1a2e')
for _,e in net.iterrows():
    a=pos[pos.passer_name==e.passer_name]; b=pos[pos.passer_name==e.receiver_name]
    if len(a)and len(b)and e.passes>=3:
        pitch.lines(a.x.values[0],a.y.values[0],b.x.values[0],b.y.values[0],
                    lw=e.passes*0.3,color='#4cc9f0',alpha=.5,ax=ax)
pitch.scatter(pos.x,pos.y,s=pos.vol*4,color='#f72585',edgecolors='white',ax=ax)
for _,p in pos.iterrows(): ax.text(p.x,p.y+2,p.passer_name.split()[-1],color='white',ha='center',fontsize=8)
ax.set_title(f"{TEAM} pass network",color='white'); plt.show()

In [ ]:
# ---- SHOT MAP, marker size = xG ----
TEAM="Spain"
shots=q(f'''SELECT start_x,start_y,xg,is_goal
            FROM `{PROJECT_ID}.statsbomb_core.fact_shot` WHERE team_name='{TEAM}' ''')
pitch=Pitch(pitch_type='statsbomb',half=True,pitch_color='#1a1a2e',line_color='#888')
fig,ax=pitch.draw(figsize=(9,7)); fig.set_facecolor('#1a1a2e')
pitch.scatter(shots.start_x,shots.start_y,s=shots.xg*600+20,
   c=shots.is_goal.map({True:'#f72585',False:'#4cc9f0'}),edgecolors='white',ax=ax)
ax.set_title(f"{TEAM} shots (size=xG, pink=goal)",color='white'); plt.show()

In [ ]:
# ---- STYLE RADAR: compare two teams ----
import numpy as np, matplotlib.pyplot as plt
A,B="Spain","England"
df=q(f'''SELECT team_name,
   AVG(directness) directness, AVG(central_progression_share) central,
   AVG(field_tilt) tilt, AVG(ppda) ppda, AVG(pass_completion_under_pressure) press_comp
   FROM `{PROJECT_ID}.statsbomb_marts.mart_team_style_labels`
   WHERE team_name IN ('{A}','{B}') GROUP BY 1''')
labels=['directness','central','tilt','ppda(inv)','press_comp']
def vals(t):
    r=df[df.team_name==t].iloc[0]
    return [r.directness,r.central,r.tilt,1/(r.ppda+0.1),r.press_comp]
ang=np.linspace(0,2*np.pi,len(labels),endpoint=False).tolist(); ang+=ang[:1]
fig,ax=plt.subplots(figsize=(7,7),subplot_kw=dict(polar=True))
for t,c in [(A,'#f72585'),(B,'#4cc9f0')]:
    v=vals(t); v+=v[:1]; ax.plot(ang,v,color=c,label=t); ax.fill(ang,v,color=c,alpha=.2)
ax.set_xticks(ang[:-1]); ax.set_xticklabels(labels); ax.legend(); plt.title(f"{A} vs {B} style"); plt.show()

## Zone analysis — the 18-zone pitch map
Where a team creates chances, loses the ball, and where they're vulnerable.

In [ ]:
# ---- ZONE HEATMAP: chance creation OR ball losses OR control ----
TEAM = "Spain"
METRIC = "shot_xg"   # try: "shot_xg", "ball_losses", "net_ball_battle", "recoveries"

import matplotlib.pyplot as plt, numpy as np
from mplsoccer import Pitch
zdf = q(f"""SELECT zone, SUM({METRIC}) val
            FROM `{PROJECT_ID}.statsbomb_marts.mart_team_zone_activity`
            WHERE team_name='{TEAM}' GROUP BY 1""")

# map zone label -> grid cell (3 thirds x 6 channels)
thirds = {"Def":0,"Mid":1,"Att":2}
chans  = {"LW":0,"LHS":1,"LC":2,"RC":3,"RHS":4,"RW":5}
grid = np.zeros((6,3))
for _,r in zdf.iterrows():
    if r.zone and "-" in r.zone:
        t,c = r.zone.split("-"); grid[chans[c], thirds[t]] = r.val

pitch = Pitch(pitch_type="statsbomb", line_color="#888", pitch_color="#1a1a2e")
fig,ax = pitch.draw(figsize=(11,7)); fig.set_facecolor("#1a1a2e")
# draw each of the 18 zones shaded by value
for ci,c in enumerate(["LW","LHS","LC","RC","RHS","RW"]):
    for ti,t in enumerate(["Def","Mid","Att"]):
        v = grid[ci,ti]; vmax = grid.max() or 1
        ax.add_patch(plt.Rectangle((ti*40, ci*13.33), 40, 13.33,
            color=plt.cm.inferno(v/vmax), alpha=0.7, zorder=0))
        ax.text(ti*40+20, ci*13.33+6.6, f"{v:.2f}", color="white",
                ha="center", fontsize=8, zorder=2)
ax.set_title(f"{TEAM} — {METRIC} by zone (attacking ->)", color="white")
plt.show()

In [ ]:
# ---- VULNERABILITY MAP: where this team is out-threatened ----
TEAM = "England"
ctrl = q(f"""SELECT zone, SUM(zone_xg_control) ctrl
             FROM `{PROJECT_ID}.statsbomb_marts.mart_team_zone_control`
             WHERE team_name='{TEAM}' GROUP BY 1 ORDER BY ctrl ASC""")
print("Most vulnerable zones (negative = conceding more than creating):")
print(ctrl.head(5).to_string(index=False))
print("\nStrongest zones:")
print(ctrl.tail(5).to_string(index=False))